# QueryVerse v0.0.1 — Text-to-SQL with a LoRA fine-tuned Llama-3.2-1B

Code for the paper **"QueryVerse: Transform Prompts into Data"** (Samuji Praneeth, Gundala Pallavi, Dr. R. Prasanna Kumar — Amrita Vishwa Vidyapeetham, Chennai).

This is the original B.Tech final-year project code, cleaned up for release. **The method is unchanged**: same base model, LoRA setup, dataset, prompt format and hyperparameters. The only changes are fixes that let it run end to end:

| Fix | Why |
|---|---|
| Removed hard-coded Hugging Face tokens | The token now comes from Colab secrets |
| Fixed the broken `pip install` line | Missing closing quote |
| Precision is detected automatically | `bf16=True` fails on T4 GPUs |
| Checkpoints on Google Drive with automatic resume | The 100-epoch run (10,900 steps) can outlast a Colab session |
| Added a `quick` run mode | Checks the whole pipeline in minutes |
| Removed duplicate, dead and empty cells | Duplicated model-loading cells, an unused MySQL install, a duplicate `dataset.map` |
| Inference uses the training chat template | The original inference prompt was raw text, unlike the chat format the model was trained on |
| Greedy decoding with `max_new_tokens` | `temperature=1.5` gave unstable SQL; `max_length=200` included the prompt |
| Evaluation cell is runnable | It referenced `generate_response`, `mistral_model` and `queryverse_model`, which were never defined |

**Runtime:** Colab → Runtime → Change runtime type → **A100** or **L4** GPU for the full run (a T4 works for quick mode).
Known limitations are listed in the repository README and addressed in **QueryVerse v0.0.2**.

## 1. Install dependencies

In [ ]:
%%capture
!pip install unsloth
!pip install rouge nltk scikit-learn gradio

## 2. Configuration
- `RUN_MODE = "quick"` trains for 60 steps to check the pipeline (~5 min). Inference and evaluation then use the model published with the paper.
- `RUN_MODE = "full"` uses the original configuration from the paper (100 epochs, 10,900 steps; the original run took 2 h 44 min of training). Use an **A100** runtime.

A full re-trained model is published to **new** repositories (`QueryVerse-v0.0.1`). The models referenced in the paper (`QueryVerse_final`, `QueryVerse_final_merged`) are left untouched.

Checkpoints are saved to Google Drive every 500 steps. If the session disconnects, re-run all cells and training resumes from the last checkpoint.

In [ ]:
RUN_MODE    = "quick"       # "quick" (pipeline check, ~25 min on T4) or "full" (paper configuration)
PUSH_TO_HUB = False         # True to publish a full re-trained model (needs HF_TOKEN in Colab secrets)
USE_DRIVE   = True          # keep checkpoints and results on Google Drive

HF_USERNAME = "EvaruRaMeeruAntha"
HF_ADAPTER  = f"{HF_USERNAME}/QueryVerse-v0.0.1"          # re-trained LoRA adapter
HF_MERGED   = f"{HF_USERNAME}/QueryVerse-v0.0.1-merged"   # re-trained merged model
PAPER_MERGED = f"{HF_USERNAME}/QueryVerse_final_merged"   # model published with the paper (not modified)

BASE_MODEL     = "unsloth/Llama-3.2-1B-Instruct"
MAX_SEQ_LENGTH = 2048
SEED           = 3407

import os
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK_DIR = "/content/drive/MyDrive/QueryVerse-v0.0.1"
else:
    WORK_DIR = "/content/QueryVerse-v0.0.1"

CHECKPOINT_DIR = f"{WORK_DIR}/checkpoints"
ADAPTER_DIR    = f"{WORK_DIR}/adapter"
RESULTS_DIR    = f"{WORK_DIR}/results"
MERGED_DIR     = "/content/QueryVerse-v0.0.1-merged"   # large; kept on the VM, published to the Hub
for d in (CHECKPOINT_DIR, ADAPTER_DIR, RESULTS_DIR):
    os.makedirs(d, exist_ok = True)

# Model used for inference, evaluation and the app
INFERENCE_MODEL = MERGED_DIR if RUN_MODE == "full" else PAPER_MERGED
print("Run mode:", RUN_MODE, "| work dir:", WORK_DIR, "| inference model:", INFERENCE_MODEL)

## 3. Load the base model (4-bit)

In [ ]:
from unsloth import FastLanguageModel, is_bfloat16_supported
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = BASE_MODEL,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = None,          # auto: float16 on T4/V100, bfloat16 on Ampere+
    load_in_4bit = True,   # 4-bit quantisation to reduce memory
)

## 4. Apply LoRA
LoRA adapters on the attention projections (`q/k/v/o_proj`), which control how tokens relate to each other, and on the MLP layers (`gate/up/down_proj`), which handle non-linear transformations.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,                     # LoRA rank
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = SEED,
    use_rslora = False,
    loftq_config = None,
)

## 5. Prepare the Spider dataset
Each (question, SQL) pair becomes a Llama-3.1 chat conversation: the question is the user turn and the SQL query is the assistant turn.

In [ ]:
from unsloth.chat_templates import get_chat_template
from datasets import load_dataset

tokenizer = get_chat_template(tokenizer, chat_template = "llama-3.1")

def format_entries(entry):
    conversations = [
        [
            {"role": "user", "content": question},
            {"role": "assistant", "content": query},
        ]
        for question, query in zip(entry["question"], entry["query"])
    ]
    texts = [
        tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False)
        for convo in conversations
    ]
    return {"text": texts}

ds = load_dataset("xlangai/spider", split = "train")
dataset = ds.map(format_entries, batched = True)
print(dataset)

In [ ]:
print(dataset[5]["text"])

## 6. Trainer
Hyperparameters are the ones used for the published model. Only the precision flags are automatic (T4 has no bfloat16), and `quick` mode limits the number of steps.

In [ ]:
import inspect
from trl import SFTTrainer, SFTConfig
from transformers import DataCollatorForSeq2Seq

full = RUN_MODE == "full"

# TRL renamed max_seq_length -> max_length in newer versions
_seq_key = "max_seq_length" if "max_seq_length" in inspect.signature(SFTConfig).parameters else "max_length"

args = SFTConfig(
    dataset_text_field = "text",
    packing = False,
    dataset_num_proc = 4,
    per_device_train_batch_size = 16,
    gradient_accumulation_steps = 4,          # effective batch size 64
    warmup_steps = 1000 if full else 5,
    num_train_epochs = 100,
    max_steps = -1 if full else 60,
    learning_rate = 1e-4,
    max_grad_norm = 1.0,
    fp16 = not is_bfloat16_supported(),
    bf16 = is_bfloat16_supported(),
    logging_steps = 50 if full else 5,
    optim = "adamw_8bit",
    weight_decay = 0.02,
    lr_scheduler_type = "cosine_with_restarts",
    seed = SEED,
    output_dir = CHECKPOINT_DIR,
    save_steps = 500,
    save_total_limit = 2,
    report_to = "none",
    **{_seq_key: MAX_SEQ_LENGTH},
)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    data_collator = DataCollatorForSeq2Seq(tokenizer = tokenizer),
    args = args,
)

In [ ]:
# Compute the loss only on the assistant's SQL, not on the question
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|start_header_id|>user<|end_header_id|>\n\n",
    response_part = "<|start_header_id|>assistant<|end_header_id|>\n\n",
)

# Check: the decoded training targets should be SQL only
sample = trainer.train_dataset[5]
print(tokenizer.decode([t for t, l in zip(sample["input_ids"], sample["labels"]) if l != -100]))

## 7. Train
Resumes automatically from the latest checkpoint on Drive if one exists (full mode only).

In [ ]:
import glob, json

checkpoints = sorted(glob.glob(f"{CHECKPOINT_DIR}/checkpoint-*"), key = lambda p: int(p.rsplit("-", 1)[1]))
resume = checkpoints[-1] if (full and checkpoints) else None
print("Resuming from:", resume or "scratch")

trainer_stats = trainer.train(resume_from_checkpoint = resume)
print(f"Training time (this session): {trainer_stats.metrics['train_runtime'] / 60:.1f} min")

# Keep the full log so the loss curve survives a disconnect
with open(f"{RESULTS_DIR}/log_history.json", "w") as f:
    json.dump(trainer.state.log_history, f, indent = 1)

## 8. Training loss curve

In [ ]:
import matplotlib.pyplot as plt

logs = trainer.state.log_history
steps = [l["step"] for l in logs if "loss" in l]
loss_values = [l["loss"] for l in logs if "loss" in l]

plt.figure(figsize = (10, 6))
plt.plot(steps, loss_values, label = "Training Loss", marker = "o")
plt.xlabel("Training Steps")
plt.ylabel("Loss")
plt.title("Training Loss Curve")
plt.legend()
plt.grid()
plt.savefig(f"{RESULTS_DIR}/training_loss.png", dpi = 150, bbox_inches = "tight")
plt.show()

## 9. Save the LoRA adapter and the merged model

In [ ]:
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print("Adapter saved to", ADAPTER_DIR)

# Merge the adapter into the base model (16-bit) for standalone inference
model.save_pretrained_merged(MERGED_DIR, tokenizer, save_method = "merged_16bit")
print("Merged model saved to", MERGED_DIR)

In [ ]:
# Optional: download the adapter as a zip
DOWNLOAD_ADAPTER = False

if DOWNLOAD_ADAPTER:
    import shutil
    from google.colab import files
    shutil.make_archive(ADAPTER_DIR, "zip", ADAPTER_DIR)
    files.download(ADAPTER_DIR + ".zip")

## 10. Publish to Hugging Face (optional)
Add a **write** token in Colab → 🔑 Secrets as `HF_TOKEN`. Never paste tokens into a cell.

In [ ]:
if PUSH_TO_HUB and RUN_MODE == "quick":
    print("Skipped: a 60-step quick-mode model should not be published.")
elif PUSH_TO_HUB:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
    model.push_to_hub(HF_ADAPTER, token = HF_TOKEN)
    tokenizer.push_to_hub(HF_ADAPTER, token = HF_TOKEN)
    model.push_to_hub_merged(HF_MERGED, tokenizer, save_method = "merged_16bit", token = HF_TOKEN)
    print(f"Published: https://huggingface.co/{HF_ADAPTER} and https://huggingface.co/{HF_MERGED}")
else:
    print("Skipped (PUSH_TO_HUB = False)")

In [ ]:
# Free GPU memory before inference
import gc
del trainer, model
gc.collect()
torch.cuda.empty_cache()

## 11. Inference
The prompt uses the same Llama-3.1 chat template as training. The schema and question go in the user turn. Decoding is greedy, and only the newly generated tokens are decoded.

In [ ]:
import re

qv_model, qv_tokenizer = FastLanguageModel.from_pretrained(
    model_name = INFERENCE_MODEL,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = None,
    load_in_4bit = False,
)
qv_tokenizer = get_chat_template(qv_tokenizer, chat_template = "llama-3.1")
FastLanguageModel.for_inference(qv_model)


def clean_sql(text):
    """Strip markdown fences / prefixes and keep the first SQL statement."""
    text = text.strip()
    fence = re.search(r"```(?:sql)?\s*(.*?)(?:```|$)", text, re.IGNORECASE | re.DOTALL)
    if fence:
        text = fence.group(1)
    text = re.sub(r"^\s*(?:sql|sql query)\s*:\s*", "", text, flags = re.IGNORECASE)
    return text.split(";")[0].strip()


def build_user_message(question, schema = None):
    if schema and schema.strip():
        schema = "\n".join(line.strip() for line in schema.strip().splitlines() if line.strip())
        return f"Schema:\n{schema}\n\nQuestion: {question.strip()}"
    return question.strip()


def generate_sql(model, tokenizer, question, schema = None, max_new_tokens = 128):
    messages = [{"role": "user", "content": build_user_message(question, schema)}]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize = True, add_generation_prompt = True,
        return_tensors = "pt", return_dict = True,
    ).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens = max_new_tokens,
            do_sample = False,
            use_cache = True,
            pad_token_id = tokenizer.eos_token_id,
        )
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return clean_sql(tokenizer.decode(new_tokens, skip_special_tokens = True))


examples = [
    "How many heads of the departments are older than 56?",
    "What are the names of the heads who are born outside the California state?",
    "Show the name and the release year of the song by the youngest singer.",
]
for q in examples:
    print("Q:  ", q)
    print("SQL:", generate_sql(qv_model, qv_tokenizer, q), "\n")

In [ ]:
db_schema = """
TABLE employees (
    id INT PRIMARY KEY,
    name VARCHAR(100),
    department VARCHAR(50),
    salary INT
);
TABLE departments (
    id INT PRIMARY KEY,
    name VARCHAR(50)
);
"""
question = "Get the names of employees who earn more than 50000."
print(generate_sql(qv_model, qv_tokenizer, question, db_schema))

## 12. Evaluation (as reported in the paper)
Compares QueryVerse with **Mistral-7B-Instruct** on the paper's example question using BLEU, ROUGE and TF-IDF cosine similarity. Both models get the same schema and question.

Note: this is a single-example, text-similarity evaluation, as in the paper. Its limitations, and the move to execution accuracy on the full Spider dev set, are covered in v0.0.2.

In [ ]:
from nltk.translate.bleu_score import sentence_bleu
from rouge import Rouge
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

sql_schema = """
CREATE TABLE Employees (
    EmployeeID INT PRIMARY KEY,
    Name VARCHAR(100),
    Age INT,
    Department VARCHAR(50),
    Salary DECIMAL(10,2),
    JoiningDate DATE
);
CREATE TABLE Departments (
    DepartmentID INT PRIMARY KEY,
    DepartmentName VARCHAR(50),
    ManagerID INT
);
"""
sql_question = ("Write an SQL query to find the average salary of employees in each department, "
                "ordered from highest to lowest.")
expected_sql_output = """
SELECT Department, AVG(Salary) AS AvgSalary
FROM Employees
GROUP BY Department
ORDER BY AvgSalary DESC;
"""


def evaluate_response(model_name, generated_sql, expected_sql):
    generated_sql = generated_sql.strip() or "<empty>"
    expected_sql = expected_sql.strip()
    bleu = sentence_bleu([expected_sql.split()], generated_sql.split())
    rouge_scores = Rouge().get_scores(generated_sql, expected_sql)[0]
    vectors = TfidfVectorizer().fit_transform([generated_sql, expected_sql])
    cosine = cosine_similarity(vectors)[0, 1]

    print(f"\n=== {model_name} ===")
    print("Generated SQL:\n", generated_sql)
    print(f"BLEU: {bleu:.4f}")
    for k in ("rouge-1", "rouge-2", "rouge-l"):
        s = rouge_scores[k]
        print(f"{k.upper():8s} P={s['p']:.4f} R={s['r']:.4f} F1={s['f']:.4f}")
    print(f"Cosine similarity: {cosine:.4f}")
    return {"model": model_name, "sql": generated_sql, "bleu": bleu,
            "rouge-1": rouge_scores["rouge-1"]["f"], "rouge-2": rouge_scores["rouge-2"]["f"],
            "rouge-l": rouge_scores["rouge-l"]["f"], "cosine": cosine}


results = [evaluate_response("QueryVerse (fine-tuned Llama-3.2-1B)",
                             generate_sql(qv_model, qv_tokenizer, sql_question, sql_schema),
                             expected_sql_output)]

In [ ]:
# Baseline: Mistral-7B-Instruct (4-bit, ~4.5 GB download)
RUN_MISTRAL_COMPARISON = True

if RUN_MISTRAL_COMPARISON:
    mistral_model, mistral_tokenizer = FastLanguageModel.from_pretrained(
        model_name = "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
        max_seq_length = MAX_SEQ_LENGTH,
        dtype = None,
        load_in_4bit = True,
    )
    FastLanguageModel.for_inference(mistral_model)
    results.append(evaluate_response("Mistral-7B-Instruct",
                                     generate_sql(mistral_model, mistral_tokenizer, sql_question, sql_schema),
                                     expected_sql_output))
    del mistral_model
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
import pandas as pd
summary = pd.DataFrame(results)
summary.to_csv(f"{RESULTS_DIR}/evaluation.csv", index = False)
summary.drop(columns = ["sql"]).round(4)

## 13. Gradio app

In [ ]:
import gradio as gr

def app_generate(schema, question):
    if not question.strip():
        return "Please enter a question."
    return generate_sql(qv_model, qv_tokenizer, question, schema)

with gr.Blocks(title = "QueryVerse") as demo:
    gr.Markdown("## QueryVerse: Natural Language to SQL")
    with gr.Row():
        schema_input = gr.Textbox(
            label = "Database schema (optional)", lines = 10,
            placeholder = "Customers(CustomerID, FirstName, LastName, Email, CreatedAt)\nOrders(OrderID, CustomerID, OrderDate)",
        )
        question_input = gr.Textbox(
            label = "Question",
            placeholder = "Show all customers who placed an order in the last 30 days.",
        )
    generate_btn = gr.Button("Generate SQL", variant = "primary")
    output_sql = gr.Code(label = "Generated SQL", language = "sql")
    generate_btn.click(app_generate, inputs = [schema_input, question_input], outputs = output_sql)

demo.launch(share = True)